# Midnight-12k Pipeline

End-to-end pipeline for the Midnight-12k encoder:
feature extraction (none / Reinhard / Macenko), paired retrieval evaluation, and UMAP visualisation.

Run on Kaggle with GPU enabled. Attach PLISM tiles as input dataset.

In [ ]:
import os, shutil, glob

# Copy project scripts to working directory
for root, dirs, files in os.walk("/kaggle/input"):
    if "plism_loader.py" in files:
        for pattern in ("*.py", "*.parquet", "*.csv"):
            for src in glob.glob(os.path.join(root, pattern)):
                shutil.copy(src, "/kaggle/working")
        break

!pip install -q --no-deps torch-staintools && pip install -q kornia umap-learn
print("Setup complete")

## 1. Feature Extraction

Extract Midnight-12k embeddings under three normalisation conditions.

In [ ]:
%%time
!python /kaggle/working/extract_features.py \
    --tiles-dir /kaggle/input/*/tiles \
    --model midnight \
    --normalise none \
    --out-dir /kaggle/working/features/midnight_none \
    --batch-size 16

In [ ]:
%%time
!python /kaggle/working/extract_features.py \
    --tiles-dir /kaggle/input/*/tiles \
    --model midnight \
    --normalise reinhard \
    --out-dir /kaggle/working/features/midnight_reinhard \
    --batch-size 16

In [ ]:
%%time
!python /kaggle/working/extract_features.py \
    --tiles-dir /kaggle/input/*/tiles \
    --model midnight \
    --normalise macenko \
    --out-dir /kaggle/working/features/midnight_macenko \
    --batch-size 16

## 2. Paired Retrieval Evaluation

Compute cross-scanner (273 pairs) and cross-staining (546 pairs) top-1 accuracy.

In [ ]:
import glob, itertools, h5py, numpy as np, pandas as pd
import torch, torch.nn.functional as F

def run_retrieval(feature_dir, run_name):
    files = sorted(glob.glob(f"{feature_dir}/*.h5"))
    assert len(files) == 91, f"Expected 91 files, found {len(files)}"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    embeddings, ref = {}, None
    for f in files:
        with h5py.File(f, "r") as h:
            s, sc = str(h.attrs["stainer"]), str(h.attrs["scanner"])
            tid = h["tile_id"][:]
            if ref is None:
                ref = tid
            embeddings[(s, sc)] = F.normalize(
                torch.from_numpy(h["features"][:]).to(device), dim=1
            )
    stainers = sorted({k[0] for k in embeddings})
    scanners = sorted({k[1] for k in embeddings})
    truth = torch.arange(len(ref), device=device)
    results = []
    with torch.inference_mode():
        for st in stainers:
            for a, b in itertools.combinations(scanners, 2):
                sim = embeddings[(st, a)] @ embeddings[(st, b)].T
                m = ((sim.argmax(1) == truth).float().mean().item()
                     + (sim.argmax(0) == truth).float().mean().item()) / 2
                results.append({"axis": "cross-scanner", "top1": m * 100})
        for sc in scanners:
            for a, b in itertools.combinations(stainers, 2):
                sim = embeddings[(a, sc)] @ embeddings[(b, sc)].T
                m = ((sim.argmax(1) == truth).float().mean().item()
                     + (sim.argmax(0) == truth).float().mean().item()) / 2
                results.append({"axis": "cross-staining", "top1": m * 100})
    df = pd.DataFrame(results)
    print(f"\n{run_name}")
    print(df.groupby("axis")["top1"].agg(["count", "mean", "std"]).round(2))
    return df

none_df = run_retrieval("/kaggle/working/features/midnight_none", "Midnight: None")
reinhard_df = run_retrieval("/kaggle/working/features/midnight_reinhard", "Midnight: Reinhard")
macenko_df = run_retrieval("/kaggle/working/features/midnight_macenko", "Midnight: Macenko")

# Save results
none_df.to_csv("/kaggle/working/retrieval_midnight_none.csv", index=False)
reinhard_df.to_csv("/kaggle/working/retrieval_midnight_reinhard.csv", index=False)
macenko_df.to_csv("/kaggle/working/retrieval_midnight_macenko.csv", index=False)
print("\nSaved retrieval CSVs")

## 3. UMAP Visualisation

Slide-level mean embeddings (91 points) projected to 2D. Produces Figure 3.

In [ ]:
import h5py
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from umap import UMAP
from itertools import combinations

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
plt.rcParams["font.size"] = 9

# Load slide-level mean embeddings
feature_dir = Path("/kaggle/working/features/midnight_none")
files = sorted(feature_dir.glob("*.h5"))

embeddings, scanners, stainers = [], [], []
for f in files:
    with h5py.File(f, "r") as h:
        embeddings.append(h["features"][:].mean(axis=0))
        scanners.append(str(h.attrs["scanner"]))
        stainers.append(str(h.attrs["stainer"]))

X = np.stack(embeddings)
print(f"Slides: {len(X)}, Embedding dim: {X.shape[1]}")

# UMAP projection
reducer = UMAP(n_neighbors=15, min_dist=0.3, random_state=42)
X_2d = reducer.fit_transform(X)

unique_scanners = sorted(set(scanners))
unique_stainers = sorted(set(stainers))

# Figure: two panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# (a) By scanner
colors_s = plt.cm.Set1(np.linspace(0, 1, len(unique_scanners)))
for i, s in enumerate(unique_scanners):
    mask = [sc == s for sc in scanners]
    pts = X_2d[mask]
    centroid = pts.mean(axis=0)
    spread = np.sqrt(((pts - centroid) ** 2).sum(axis=1).mean())
    ax1.scatter(pts[:, 0], pts[:, 1], s=20, c=[colors_s[i]], alpha=0.3, zorder=2)
    ax1.scatter(centroid[0], centroid[1], s=spread * 300, c=[colors_s[i]],
               label=f"{s} ({spread:.1f})", alpha=0.7,
               edgecolors="white", linewidth=1.2, zorder=3)

ax1.set_xlabel("UMAP 1")
ax1.set_ylabel("UMAP 2")
ax1.set_title("(a) Coloured by scanner", fontweight="bold", loc="left")
ax1.legend(title="Scanner (spread)", fontsize=7, loc="upper right")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# (b) By staining condition
colors_t = plt.cm.tab20(np.linspace(0, 1, len(unique_stainers)))
for i, s in enumerate(unique_stainers):
    mask = [st == s for st in stainers]
    pts = X_2d[mask]
    centroid = pts.mean(axis=0)
    spread = np.sqrt(((pts - centroid) ** 2).sum(axis=1).mean())
    ax2.scatter(pts[:, 0], pts[:, 1], s=20, c=[colors_t[i]], alpha=0.3, zorder=2)
    ax2.scatter(centroid[0], centroid[1], s=spread * 300, c=[colors_t[i]],
               label=f"{s} ({spread:.1f})", alpha=0.7,
               edgecolors="white", linewidth=1.2, zorder=3)

ax2.set_xlabel("UMAP 1")
ax2.set_ylabel("UMAP 2")
ax2.set_title("(b) Coloured by staining condition", fontweight="bold", loc="left")
ax2.legend(title="Stainer (spread)", fontsize=6, loc="upper right", ncol=2,
           handletextpad=0.3, columnspacing=0.5)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# Stats
scanner_centroids = {s: X_2d[[sc == s for sc in scanners]].mean(axis=0)
                     for s in unique_scanners}
dists = [np.linalg.norm(scanner_centroids[a] - scanner_centroids[b])
         for a, b in combinations(unique_scanners, 2)]
scanner_spreads = [np.sqrt(((X_2d[[sc == s for sc in scanners]]
                             - scanner_centroids[s]) ** 2).sum(axis=1).mean())
                   for s in unique_scanners]
print(f"Mean inter-centroid distance: {np.mean(dists):.2f}")
print(f"Mean within-group spread: {np.mean(scanner_spreads):.2f}")

plt.tight_layout()
fig.savefig("/kaggle/working/figure3_umap.pdf", format="pdf", bbox_inches="tight")
fig.savefig("/kaggle/working/figure3_umap.png", dpi=200, bbox_inches="tight")
print("Saved figure3_umap.pdf and .png")